In [ ]:
from demo import *

print("ml_analysis demo — end-to-end pipeline")

# --- Regenerate synthetic data ---
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)

rng = np.random.default_rng(RANDOM_SEED)
assets = ["A01", "A02", "A03"]
classes = ["TP", "FP", "TN", "FN"]
replacement_types = ["bearing", "seal"]

print(f"\n[1/5] Generating synthetic data  ({len(assets)} assets × {len(classes)} classes"
        f" × {N_EVENTS_PER_CLASS} events each) ...")
labels = generate_synthetic_data(
    assets=assets,
    classes=classes,
    replacement_types=replacement_types,
    n_per_class=N_EVENTS_PER_CLASS,
    event_len_h=EVENT_LEN_HOURS,
    rng=rng,
)
print(f"   Label table: {labels.shape[0]} events")

print("\n[2/5] Registering features ...")
register_features()

print("\n[3/5] Building event dataset + materialising period aggregates ...")
events = build(labels, cfg=cfg)
period = to_period(
    events,
    cfg=cfg,
    aggregators=["mean", "std", "min", "max", "p05", "p95"],
)
print(f"   Period table: {period.shape[0]} rows × {period.shape[1]} columns")





In [ ]:
print("\n[4/5] Running analysis suite ...")
ctx = AnalysisContext(
    df=period,
    cfg=cfg,
    target_col="class",
    label_filter={"class": ["TP", "FP", "TN", "FN"]},
    stratify_by="replacement_type",
    output_dir=str(OUTPUT_DIR),
)

analyses = [
    DistributionAnalysis(),
    PairwiseSeparability(top_n=10),
    FeatureImportance(
        rf_params={"n_estimators": 200, "n_jobs": -1, "random_state": RANDOM_SEED},
        permutation_repeats=5,
    ),
    ClusterAnalysis(),
    ClassifierEvaluation(run_lgb=True, run_xgb=True),
    Stratified(
        inner=FeatureImportance(
            name="importance_strat",
            rf_params={"n_estimators": 100, "n_jobs": -1, "random_state": RANDOM_SEED},
            permutation_repeats=3,
        ),
        by="replacement_type",
    ),
]
results = run_analyses(analyses, ctx)

In [ ]:
cluster = results["clustering"]

print(f"Best k (KMeans): {cluster['best_k']}")
print(f"Algorithms fitted: {list(cluster['labels'])}")
print(f"Reductions available: {list(cluster['reductions'])}")
print(f"Class names: {cluster['class_names']}")

print_clustering(cluster)

In [ ]:
# 2-D projection coloured by true class vs. by KMeans assignment
import matplotlib.pyplot as plt
import numpy as np

reduction_name = "UMAP" if "UMAP" in cluster["reductions"] else "PCA"
emb = cluster["reductions"][reduction_name]
y_true = cluster["y_true"]  # encoded true labels, row-aligned with the embeddings
kmeans_key = next(k for k in cluster["labels"] if k.startswith("KMeans"))
y_km = cluster["labels"][kmeans_key]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (labels, title) in zip(
    axes,
    [(y_true, f"{reduction_name} — true class"),
     (y_km, f"{reduction_name} — {kmeans_key}")],
):
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=labels, cmap="tab10", s=18, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel(f"{reduction_name}-1"); ax.set_ylabel(f"{reduction_name}-2")
plt.tight_layout(); plt.show()

### Cluster composition vs. true classes

The scatter shows *where* points land in 2-D; this heatmap quantifies the
overlap. Each panel is one clustering algorithm: rows are clusters, columns are
the true classes, the colour is the fraction of the cluster falling in each class
(so every row sums to 1) and the number is the raw count. A row dominated by a
single dark cell means that cluster cleanly recovers one class; a row spread
evenly across columns means the cluster cuts across classes. Noise points
(`-1` from DBSCAN / HDBSCAN) are excluded, matching the alignment metrics.

In [ ]:
# Cluster-vs-true-class contingency, one panel per algorithm.
from ml_analysis.io.stat_plots import cluster_class_heatmap_panel

fig = cluster_class_heatmap_panel(
    cluster["labels"],
    cluster["y_true"],
    cluster["class_names"],
)
plt.show()

## Cluster Analysis

`ClusterAnalysis` runs **KMeans** (with auto-selected `k`), **DBSCAN**, and **HDBSCAN**
on the standardised feature matrix, then reduces to 2-D with **PCA** (and **UMAP**
if installed) for visualisation. Each clustering is scored against the true class
labels with silhouette / Davies-Bouldin (intrinsic) and ARI / NMI / V-measure
(alignment with the supervised labels).

In [ ]:
print("\n[5/5] Results")
print_distributions(results["distributions"])
print_pairwise(results["pairwise"])
print_importance(results["importance"])
print_clustering(results["clustering"])
print_classifier(results["classifier"])
print_stratified(results["stratified__importance_strat"])


save_summary_figure(
    importance_result=results["importance"],
    distributions_result=results["distributions"],
    output_dir=OUTPUT_DIR,
)

print("\nDone.\n")

## Within-event dynamics with `to_windowed`

`to_period` collapses each event to a single row of summary statistics — great for tabular ML, but it discards the *shape* of the event over time.

`to_windowed` keeps that shape: it groups each event into fixed-width time windows and aggregates within each, returning **N rows per event** instead of one. Useful for sequence models, change-point detection, or simply visualising how a signal evolves before/during/after an anomaly.

Below we take one example event per class, slice each into 15-minute tumbling windows, and plot the per-window mean of `temperature` with a ±1 std band. The injected TP/FP bursts should appear as a hump in the middle of the event; TN/FN should look flat.

In [ ]:
import polars as pl
from ml_analysis.features.materialize import to_windowed

# Pick one example event per class straight from the dict returned by `build`.
sample_events: dict[str, pl.LazyFrame] = {}
for evid, lf in events.items():
    cls = lf.first().collect()["class"][0]
    sample_events.setdefault(cls, lf)
    if len(sample_events) == 4:
        break

# 15-min tumbling windows on the raw signals.
# feature_names=[] skips the rolling features so the demo output stays compact.
windowed_frames = []
for cls, lf in sample_events.items():
    w = to_windowed(
        lf,
        cfg=cfg,
        every="15m",
        period="15m",
        sources=["temperature", "vibration", "pressure"],
        aggregators=["mean", "std"],
        feature_names=[],
    ).collect()
    windowed_frames.append(w)

windowed = pl.concat(windowed_frames)
n_windows = windowed.height // len(sample_events)
print(f"Windowed table: {windowed.height} rows  "
      f"({len(sample_events)} events × {n_windows} windows of 15 min)\n")

print(windowed.select([
    "event_id", "class", "timestamp",
    "temperature__mean", "temperature__std",
]).head(8))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, cls in zip(axes, ["TP", "FP", "TN", "FN"]):
    if cls not in sample_events:
        ax.set_visible(False)
        continue
    sub = windowed.filter(pl.col("class") == cls).sort("timestamp")
    ts = sub["timestamp"].to_numpy()
    t_min = (ts - ts[0]) / np.timedelta64(1, "m")
    mean = sub["temperature__mean"].to_numpy()
    std = sub["temperature__std"].to_numpy()
    ax.plot(t_min, mean, color="steelblue", linewidth=2, label="window mean")
    ax.fill_between(t_min, mean - std, mean + std, color="steelblue", alpha=0.25, label="±1 std")
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_title(f"class = {cls}")
    ax.set_xlabel("Minutes since event start")

axes[0].set_ylabel("temperature  (15-min window stats)")
axes[0].legend(loc="upper right", fontsize=9)
fig.suptitle("Within-event dynamics  —  to_windowed(every=15m, period=15m)")
fig.tight_layout()
plt.show()

## Corroborating statistical tests

The original pipeline already ranks features and trains classifiers. This section
demonstrates the **corroboration layer**: multiple-testing correction, additional
effect sizes / pairwise tests, bootstrap CIs, cross-validated metrics, importance
stability, and cluster validation.

See `STATISTICAL_TESTS.md` for what each test does and how to read the output.


### 1. Distribution analysis with multiple-testing correction

`DistributionAnalysis` now reports Kruskal-Wallis, ANOVA F, Anderson-Darling, and
Levene's test with BH-FDR and Bonferroni corrections. A feature with
`kw_p_bh_fdr < 0.05` *and* a strong `cohens_d` is unambiguous; everything else
deserves scrutiny.


In [ ]:
dist_summary = results["distributions"]["summary"]
cols = [c for c in [
    "feature", "kw_stat", "kw_p", "kw_p_bh_fdr", "anova_p_bh_fdr",
    "ad_p", "levene_p",
] if c in dist_summary.columns]
dist_summary[cols].head(10).round(4)


### 2. Pairwise battery + bootstrap AUC CI

`PairwiseSeparability(bootstrap_n=N)` adds Mann-Whitney U, Brunner-Munzel,
Welch's t, Cohen's d, Hedges' g, Wasserstein, JS divergence, and bootstrap CIs
for AUC and Cliff's delta.

The **volcano plot** (left) puts every feature on (|effect size|, -log10 corrected p);
the top-right quadrant contains real discriminators. The **AUC ± CI bar chart**
(right) confirms that the top features' AUC bootstrap interval does not cross 0.5.


In [ ]:
from ml_analysis.io.stat_plots import volcano_plot, auc_bootstrap_plot
import matplotlib.pyplot as plt

def _find_pair(pairs, *wanted):
    target = set(wanted)
    for k in pairs:
        if set(k) == target:
            return k
    return next(iter(pairs))

pair = _find_pair(results["pairwise"]["pairs"], "FP", "TP")
pair_tbl = results["pairwise"]["pairs"][pair]
pair_label = f"{pair[0]} vs {pair[1]}"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
volcano_plot(pair_tbl, pair_label=pair_label, ax=axes[0])
auc_bootstrap_plot(pair_tbl, top_n=10, pair_label=pair_label, ax=axes[1])
fig.tight_layout(); plt.show()

# Inspect the full extended-battery table
cols = [c for c in [
    "feature", "auc", "auc_ci_low", "auc_ci_high",
    "cliffs_delta", "cohens_d", "wasserstein",
    "mwu_p_bh_fdr", "ks_p_bh_fdr",
] if c in pair_tbl.columns]
pair_tbl[cols].head(8).round(3)


### 3. Importance stability

`ImportanceStability` bootstraps the random forest to get a percentile CI on each
feature's MDI and a top-k stability fraction. The companion `method_agreement`
matrix shows how strongly RF MDI / permutation / ANOVA F / Kruskal-Wallis /
mutual information agree (Spearman ρ). High agreement off-diagonal corroborates
the composite importance ranking.


In [ ]:
from ml_analysis.io.stat_plots import importance_stability_plot, method_agreement_heatmap

stab = results["importance_stability"]
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
importance_stability_plot(stab["bootstrap_table"], top_n=10, ax=axes[0])
method_agreement_heatmap(stab["method_agreement"], ax=axes[1])
fig.tight_layout(); plt.show()

stab["bootstrap_table"].head(10).round(4)


### 4. Cross-validated classifier

A stratified k-fold replacement for the single-split `ClassifierEvaluation`.
Reports balanced accuracy, MCC, Cohen's kappa, ROC-AUC, PR-AUC, Brier, ECE —
each with per-fold spread. A small `std` confirms the headline `mean` is real.


In [ ]:
from ml_analysis.io.stat_plots import cv_metric_boxplot

cv = results["cv_classifier"]
fig, ax = plt.subplots(figsize=(9, 5))
cv_metric_boxplot(cv["per_fold"], ax=ax)
fig.tight_layout(); plt.show()

cv["summary"].round(3)


### 5. Calibration curve  (binary subset)

The CV out-of-fold probabilities power a reliability diagram. We restrict to
TP vs FP because calibration is naturally defined for binary problems.


In [ ]:
from ml_analysis.io.stat_plots import calibration_plot
from ml_analysis.analysis import CrossValidatedClassifier
from ml_analysis.analysis.base import prepare_xy

binary_ctx = AnalysisContext(
    df=period, cfg=cfg, target_col="class",
    label_filter={"class": ["TP", "FP"]},
)
binary_cv = CrossValidatedClassifier(
    n_splits=5,
    rf_params={"n_estimators": 200, "n_jobs": -1, "random_state": RANDOM_SEED},
).run(binary_ctx)

y = prepare_xy(binary_ctx).y
fig, ax = plt.subplots(figsize=(5, 5))
calibration_plot(y, binary_cv["oof_proba"][:, 1], n_bins=10, ax=ax)
fig.tight_layout(); plt.show()


### 6. Cluster validation — Hopkins + permutation test

`ClusterValidation` adds three things on top of `ClusterAnalysis`:

- **Hopkins statistic** — is the data clusterable at all?  ≈0.5 = no, ≈1 = strongly.
- **Calinski-Harabasz** — a third internal validity score next to silhouette and
  Davies-Bouldin.
- **Permutation p-values for ARI / V-measure** — by shuffling the class labels we
  derive a null distribution and decide whether the observed alignment between
  KMeans clusters and class labels is real.


In [ ]:
results["cluster_validation"]["summary"].round(4)


In [ ]:
# Permutation null distribution for ARI (re-derived for visualisation)
import numpy as np
from sklearn.metrics import adjusted_rand_score
from ml_analysis.io.stat_plots import permutation_null_plot

cv_res = results["cluster_validation"]
labels = cv_res["labels_used"]
y = prepare_xy(ctx).y
mask = np.asarray(labels) != -1
y_kept, lab_kept = y[mask], np.asarray(labels)[mask]

rng = np.random.default_rng(cfg.random_state)
null = np.empty(400)
perm = y_kept.copy()
for i in range(len(null)):
    rng.shuffle(perm)
    null[i] = adjusted_rand_score(perm, lab_kept)

s = cv_res["summary"].iloc[0]
fig, ax = plt.subplots(figsize=(7, 4))
permutation_null_plot(null, observed=float(s["ari"]),
                      p_value=float(s["ari_perm_p"]),
                      statistic_name="ARI", ax=ax)
fig.tight_layout(); plt.show()


### 7. Recipe for a robust feature claim

A feature is "really useful" to separate two classes when **all** of these agree:

1. `distributions.summary`: `kw_p_bh_fdr < 0.05` *and* `anova_p_bh_fdr < 0.05`
2. `pairwise.pairs[pair]`: AUC CI excludes 0.5, |Cohen's d| > 0.5, `mwu_p_bh_fdr < 0.05`
3. `importance_stability`: `mdi_ci_low > 0` and `stability_top10 > 0.8`; methods agree (ρ > 0.7)
4. `cv_classifier`: MCC and balanced accuracy both meaningfully above chance, with small `std`
5. `cluster_validation`: `hopkins > 0.6` and `ari_perm_p < 0.05`

When two of these disagree, the claim should be **softened**, not strengthened —
disagreement is informative.


## v0.1 capabilities: the `Run` facade

Everything above used the low-level `AnalysisContext` API. The `Run`
facade wraps it with per-analysis shortcuts, shared preparation
caching, and persistence:

- **`separability()`** — permutation-tested CV: *can the classes be
  discriminated at all?*
- **`anomaly()`** — unsupervised IsolationForest + LOF + Mahalanobis
  ensemble, fit on a healthy baseline, with per-feature attribution.
- **`label_spreading()` / `pu_learning()`** — semi-supervised modes
  for sparsely-labeled data.
- **`changepoint()` / `lagged_relations()` / `mi_network()` /
  `correlation_structure()`** — unsupervised structure discovery.
- **`save()` / `report()`** — parquet + manifest run directory
  (browsable with the Streamlit dashboard) and a self-contained HTML
  report.


In [ ]:
from ml_analysis import Run

run = Run(period, target_col="class", cfg=cfg)

# Is class discrimination possible at all?
sep = run.separability(n_permutations=200,
                       rf_params={"n_estimators": 100, "n_jobs": -1})
display(sep.frames["summary"].round(4))

# Unsupervised anomaly scores, fit on the quiet TN/FN baseline
ano = run.anomaly(baseline_filter={"class": ["TN", "FN"]})
scores = ano.frames["scores"].merge(
    period.select("event_id", "class").to_pandas(), on="event_id")
display(scores.groupby("class")["ensemble"].mean().round(3).to_frame())

# Redundant-channel structure
corr = run.correlation_structure()
display(corr.frames["duplicates"].head(5).round(3))

# Persist: parquet run dir (dashboard-ready) + standalone HTML report
run_dir = run.save(OUTPUT_DIR / "runs", name="notebook_run")
report = run.report(OUTPUT_DIR / "notebook_report.html")
print(f"run dir → {run_dir}")
print(f"report  → {report}")
print(f"dashboard: streamlit run src/ml_analysis/dashboard/app.py -- --root {OUTPUT_DIR / 'runs'}")
